## Gemma-2 Fine-Tuning Notebook for Poem Generation

Some code snippets ideas was taken from https://docs.unsloth.ai/get-started/unsloth-notebooks

This notebook demonstrates how to fine-tune the Gemma-2 2B parameter language model to generate poems using the Unsloth framework for efficient training. The implementation includes data preparation, model training, evaluation, and inference capabilities.

For efficient fine-tuning, the following methods were used:

- LoRA (Low-Rank Adaptation): Reduces the number of trainable parameters by adding low-rank matrices to the model layers, making fine-tuning more efficient.

- 4-bit Quantization: Compresses the model weights to 4 bits, significantly reducing memory usage while maintaining performance.


### Installation and Setup
- Installs Unsloth library for efficient fine-tuning

In [ ]:
import os
%pip install unsloth -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# use this if you run in own pc or kaggle etc
# !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo -q
# !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer -q
# !pip install --no-deps unsloth -q

### Unsloth Framework Usage
This framework provides optimized implementations for faster and more memory-efficient fine-tuning of large language models. It includes features like gradient checkpointing, mixed-precision training, and efficient memory management, which are particularly useful for resource-constrained environments like Colab.

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = (
    None
)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Gemma2 patching. Transformers: 4.50.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters for the following target modules:

- **q_proj, k_proj, v_proj, o_proj**: These are the query, key, value, and output projection layers in the attention mechanism.

- **gate_proj, up_proj, down_proj**: These are the feed-forward network layers in the transformer blocks.

These targets were chosen because they are critical for the model's ability to process and generate text, and adapting them with LoRA allows for efficient fine-tuning for poem generation without modifying the entire model.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.3.19 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


<a name="Data Preparation"></a>
### Data Preparation

- Downloads the cutom dataset that was collected from https://www.wishafriend.com/poems/
- Determines system prompt for a LLM
- Adds EOS token to aboid infinite generations
- Formats the data

In [6]:
greeting_prompt = """Below is a theme and title for a poem. Write a greeting poem that matches the theme and title for a greeting postcard.

### Theme:
{}

### Title:
{}

### Poem:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    themes = examples["Theme"]
    titles = examples["Title"]
    poems = examples["Poem"]
    texts = []
    for theme, title, poem in zip(themes, titles, poems):
        text = greeting_prompt.format(theme, title, poem) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }
pass

from datasets import load_dataset
dataset = load_dataset("csv", data_files = "/content/poems.csv", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [7]:
from datasets import Dataset

dataset = dataset.shuffle(seed=42)
split = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split['train']
eval_dataset = split['test']

### Evaluation metric

We used BERTScore for the poem generation task because:
- **Semantic Relevance**: Unlike BLEU or ROUGE, BERTScore leverages contextual embeddings from pre-trained language models (like BERT), which allows it to better capture the semantic meaning of the generated poem compared to the reference.
- **Flexibility in Wording**: In poetry, creative phrasing and word choices are common. BERTScore accommodates such variations better, as it does not rely on exact n-gram matches.

In [ ]:
%pip install evaluate bert_score -q

In [8]:
from evaluate import load
import numpy as np

bertscore = load("bertscore")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Decode token IDs to strings
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Clean up spacing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Compute BERTScore
    results = bertscore.compute(predictions=decoded_preds,
                                references=decoded_labels,
                                lang="en")

    # Return average F1 score
    return {
        "bertscore_f1": np.mean(results["f1"])
    }

<a name="Train"></a>
### Train the model
Here we used TRL SFFTrainer because


#### Using wandb to monitor fine-tuning

In [9]:
%pip install wandb -q

In [10]:
import wandb
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pupyrkavalensia (pupyrkavalensia-innopolis-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

#### Training Part

In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        # num_train_epochs = 1,
        max_steps=60,
        learning_rate=3e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="wandb",
    ),
)

In [14]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
2.162 GB of memory reserved.


In [15]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,345 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 10,383,360/2,000,000,000 (0.52% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: pupyrkavalensia (pupyrkavalensia-innopolis-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.707100
2,2.753900
3,2.737500
4,2.466300
5,2.178500
6,2.021800
7,1.862200
8,1.871100
9,1.715000
10,1.692700


In [ ]:
metrics = trainer_stats.evaluate()
print(metrics)

In [17]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

215.4206 seconds used for training.
3.59 minutes used for training.
Peak reserved memory = 4.139 GB.
Peak reserved memory for training = 1.977 GB.
Peak reserved memory % of max memory = 28.078 %.
Peak reserved memory for training % of max memory = 13.412 %.


<a name="Inference"></a>
### Inference
We' run the model and look at few exmples of poem generations

In [ ]:
# Enable faster inference
FastLanguageModel.for_inference(model)

# Prepare the input
instruction = "Write a poem for my mother's day"
themes = "appreciation, love, sucess"

inputs = tokenizer(
    greeting_prompt.format(
        instruction,
        themes,
        "",
    ),
    return_tensors="pt"
).to("cuda")

# Generate the poem with improved parameters
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
    use_cache=True
)

# Decode and clean the output
generated_poem = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# Extract just the generated poem part (after "### Poem:")
poem_start = generated_poem.find("### Poem:") + len("### Poem:")
final_poem = generated_poem[poem_start:].strip()

print("Generated Poem:")
print(final_poem)

Generated Poem:
You have done so much in life 

With your will you can achieve anything 

Mom I am proud of you from heart 

You are always there to make me strong 

Your hard work has paid off through time 

And now things are not same for me 

I just want to say thank you mom 

For everything!


In [26]:
FastLanguageModel.for_inference(model)

instruction = "Write a romantic birthday poem for my girlfriend"
themes = "beauty, love, dreams"

# Generate the poem with improved parameters
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
    use_cache=True
)

# Decode and clean the output
generated_poem = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# Extract just the generated poem part (after "### Poem:")
poem_start = generated_poem.find("### Poem:") + len("### Poem:")
final_poem = generated_poem[poem_start:].strip()

print("Generated Poem:")
print(final_poem)

Generated Poem:
Your beauty has always been a dream come true 

And every moment with you is so beautiful

I have fallen in love with your eyes 

The way they sparkle when I see them shine 

You are truly special to me dear

My baby girl, I wish you a very happy birthday!


In [27]:
FastLanguageModel.for_inference(model)

instruction = "Create a heartfelt poem for my grandma's 80th birthday"
themes = "wisdom, family, tenderness"

# Generate the poem with improved parameters
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
    use_cache=True
)

# Decode and clean the output
generated_poem = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# Extract just the generated poem part (after "### Poem:")
poem_start = generated_poem.find("### Poem:") + len("### Poem:")
final_poem = generated_poem[poem_start:].strip()

print("Generated Poem:")
print(final_poem)

Generated Poem:
Wisdom from you was always there in life, 

I have been blessed to be your daughter. 

You are very sweet like honey, 

And I wish you get all happiness this way!


In [28]:
FastLanguageModel.for_inference(model)

# Prepare the input
instruction = "Generate a Father’s Day poem for my dad"
themes = "strength, guidance, pride"

# Generate the poem with improved parameters
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
    use_cache=True
)

# Decode and clean the output
generated_poem = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# Extract just the generated poem part (after "### Poem:")
poem_start = generated_poem.find("### Poem:") + len("### Poem:")
final_poem = generated_poem[poem_start:].strip()

print("Generated Poem:")
print(final_poem)

Generated Poem:
Dad you are strong 

You have been guiding me since I was young 

It's all because of your love in life 

That everything gets sorted in time 

I just want to thank you so much 

For being there when I needed you the most 

Happy Fathers day to you!


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model we use Huggingface's `push_to_hub` for an online save.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [25]:
from huggingface_hub import create_repo

create_repo("poem-gemma2-lora-model", private=False)

RepoUrl('https://huggingface.co/Nazgulitos/poem-gemma2-lora-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Nazgulitos/poem-gemma2-lora-model')

In [22]:
model.save_pretrained("/content/drive/MyDrive/poem_model/lora_model")  # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/poem_model/lora_model")

('/content/drive/MyDrive/poem_model/lora_model/tokenizer_config.json',
 '/content/drive/MyDrive/poem_model/lora_model/special_tokens_map.json',
 '/content/drive/MyDrive/poem_model/lora_model/tokenizer.model',
 '/content/drive/MyDrive/poem_model/lora_model/added_tokens.json',
 '/content/drive/MyDrive/poem_model/lora_model/tokenizer.json')

In [29]:
model.push_to_hub("Nazgulitos/poem-gemma2-lora-model") # Online saving
tokenizer.push_to_hub("Nazgulitos/poem-gemma2-lora-model") # Online saving

adapter_model.safetensors:   0%|          | 0.00/41.6M [00:00<?, ?B/s]

Saved model to https://huggingface.co/Nazgulitos/poem-gemma2-lora-model


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [30]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "poem-gemma2-lora-model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

greeting_prompt = """Below is a theme and title for a poem. Write a greeting poem that matches the theme and title for a greeting postcard.

### Theme:
{}

### Title:
{}

### Poem:
{}"""

inputs = tokenizer(
    greeting_prompt.format(
        "Write a sweet poem to my best friend to cheer her up",
        "support, happiness, shared memories",
        "",
    ),
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
    use_cache=True
)

generated_poem = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
poem_start = generated_poem.find("### Poem:") + len("### Poem:")
final_poem = generated_poem[poem_start:].strip()

print("Generated Poem:")
print(final_poem)

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer)
# _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

Generated Poem:
You are always there in every situation 

It's like you know me from start 

I am so lucky I have found a true friend 

Who cares about everything in life 

The love that we share between us 

Is just too amazing and sublime 

My friendship with you will remain forever 

As long as this world exists!


We can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # If Unsloth is not possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "poem-gemma2-lora-model",
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")